# Hugging Face 기초 실습

이번 노트북은 프로젝트를 바로 해결하기 전에 Hugging Face의 기본 흐름을 익히기 위한 실습입니다.

오늘의 핵심 흐름은 아래 한 줄입니다.

```text
문장 데이터 -> tokenizer로 숫자 변환 -> model에 입력 -> 예측 또는 학습
```

처음부터 전체 학습을 돌리기보다, 작은 예제로 하나씩 확인하면서 진행합니다.

## STEP 0. 라이브러리 확인

먼저 실습에 필요한 라이브러리가 현재 환경에서 잘 불러와지는지 확인합니다.

- `transformers`: 모델, tokenizer, pipeline을 제공하는 라이브러리
- `datasets`: Hugging Face 데이터셋을 쉽게 불러오는 라이브러리
- `torch`: 딥러닝 모델 계산에 사용하는 라이브러리

여기서는 버전 숫자를 외울 필요는 없습니다. import가 잘 되는지만 먼저 확인하면 됩니다.

In [1]:
# 실습에 필요한 라이브러리를 불러옵니다.
# 에러 없이 실행되면 기본 준비가 된 상태입니다.

import transformers
import datasets
import torch

print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__)
print('torch:', torch.__version__)
print('cuda 사용 가능:', torch.cuda.is_available())

c:\Users\Administrator\Desktop\codex-workspace\envs\AIFFEL_quest_eng_py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.8.1
datasets: 4.8.5
torch: 2.11.0+cu128
cuda 사용 가능: True


### 확인하기

위 셀에서 에러가 나지 않으면 다음 단계로 넘어갑니다.

`cuda 사용 가능`이 `False`여도 괜찮습니다. GPU가 없다는 뜻이고, 작은 실습 코드는 CPU에서도 실행할 수 있습니다.

## STEP 1. pipeline으로 모델 사용해보기

`pipeline`은 Hugging Face 모델을 가장 쉽게 써보는 방법입니다.

복잡한 tokenizer와 model 코드를 직접 작성하지 않아도, 문장을 넣으면 바로 예측 결과를 볼 수 있습니다.

여기서는 영어 감성분석 모델을 사용해서 문장이 긍정인지 부정인지 확인해봅니다.

In [2]:
from transformers import pipeline

# sentiment-analysis는 문장의 감정을 분류하는 작업입니다.
# 처음 실행할 때는 모델 파일을 다운로드하느라 시간이 걸릴 수 있습니다.
sentiment_pipeline = pipeline('sentiment-analysis')

result = sentiment_pipeline('I love this movie!')
result

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8089.15it/s]


[{'label': 'POSITIVE', 'score': 0.9998775720596313}]

In [3]:
# 문장을 몇 개 더 넣어봅니다.
# 리스트를 넣으면 여러 문장을 한 번에 예측할 수 있습니다.

sentences = [
    'I love this movie!',
    'This movie was boring and too long.',
    'The acting was great, but the story was confusing.'
]

sentiment_pipeline(sentences)

[{'label': 'POSITIVE', 'score': 0.9998775720596313},
 {'label': 'NEGATIVE', 'score': 0.9997715353965759},
 {'label': 'NEGATIVE', 'score': 0.9938702583312988}]

### 확인하기

출력 결과는 보통 아래와 비슷한 형태입니다.

```python
[{'label': 'POSITIVE', 'score': 0.999...}]
```

- `label`: 모델이 예측한 정답
- `score`: 모델이 그 예측을 얼마나 확신하는지 나타내는 값

아직 학습을 한 것은 아닙니다. 이미 학습된 모델을 가져와서 사용해본 것입니다.

## STEP 2. Hugging Face 데이터셋 불러오기

모델을 학습하려면 데이터가 필요합니다.

Hugging Face의 `datasets` 라이브러리를 사용하면 공개 데이터셋을 쉽게 불러올 수 있습니다.

이번 실습에서는 GLUE benchmark의 MRPC 데이터셋을 살펴봅니다. MRPC는 두 문장이 의미상 같은지 다른지 판단하는 데이터셋입니다.

In [4]:
from datasets import load_dataset

# glue 데이터셋 안의 mrpc 태스크를 불러옵니다.
# 처음 실행할 때는 데이터 다운로드 시간이 걸릴 수 있습니다.
mrpc = load_dataset('glue', 'mrpc')

mrpc

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
# 데이터셋은 train, validation, test처럼 용도별로 나뉘어 있습니다.
# train은 학습용, validation은 학습 중 확인용, test는 최종 평가용이라고 이해하면 됩니다.

print(mrpc.keys())
print('train 개수:', len(mrpc['train']))
print('validation 개수:', len(mrpc['validation']))
print('test 개수:', len(mrpc['test']))

dict_keys(['train', 'validation', 'test'])
train 개수: 3668
validation 개수: 408
test 개수: 1725


In [6]:
# train 데이터의 첫 번째 샘플을 확인합니다.
# sentence1, sentence2는 비교할 두 문장입니다.
# label은 두 문장의 관계를 나타내는 정답입니다.

mrpc['train'][0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [7]:
# 컬럼 이름과 feature 정보를 확인합니다.
# label의 ClassLabel 정보를 보면 0과 1이 어떤 의미인지 알 수 있습니다.

print(mrpc['train'].column_names)
mrpc['train'].features

['sentence1', 'sentence2', 'label', 'idx']


{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

### 여기까지 정리

이번 단계에서는 아직 모델을 직접 학습하지 않았습니다.

대신 아래 내용을 확인했습니다.

1. Hugging Face 라이브러리를 불러올 수 있다.
2. `pipeline`으로 이미 학습된 모델을 바로 사용할 수 있다.
3. `datasets`로 학습용 데이터를 불러올 수 있다.
4. 데이터셋은 보통 train/validation/test로 나뉜다.

다음 단계에서는 tokenizer가 문장을 숫자로 바꾸는 과정을 직접 확인합니다.

## STEP 3. tokenizer로 문장을 숫자로 바꾸기

딥러닝 모델은 문장을 글자 그대로 읽지 못합니다.

모델은 숫자로 된 텐서를 입력받아 계산하므로, 문장을 먼저 숫자 ID의 배열로 바꿔야 합니다. 이 일을 하는 도구가 `tokenizer`입니다.

이번 단계에서는 MRPC 데이터의 `sentence1`, `sentence2`를 tokenizer에 넣고, 모델 입력이 어떤 형태로 바뀌는지 확인합니다.

In [8]:
from transformers import AutoTokenizer

# AutoTokenizer는 모델 이름에 맞는 tokenizer를 자동으로 불러옵니다.
# distilbert-base-uncased는 영어 문장을 처리하는 비교적 가벼운 BERT 계열 모델입니다.
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

tokenizer

c:\Users\Administrator\Desktop\codex-workspace\envs\AIFFEL_quest_eng_py312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Administrator\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [9]:
# MRPC train 데이터에서 샘플 하나를 꺼내봅니다.
# 이 데이터는 두 문장(sentence1, sentence2)이 같은 의미인지 판단하는 문제입니다.

sample = mrpc['train'][0]

print('sentence1:', sample['sentence1'])
print('sentence2:', sample['sentence2'])
print('label:', sample['label'])

sentence1: Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
sentence2: Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .
label: 1


In [10]:
# tokenizer에 두 문장을 함께 넣습니다.
# truncation=True는 문장이 너무 길면 모델이 받을 수 있는 길이에 맞게 자르겠다는 의미입니다.

encoded = tokenizer(
    sample['sentence1'],
    sample['sentence2'],
    truncation=True
)

encoded

{'input_ids': [101, 2572, 3217, 5831, 5496, 2010, 2567, 1010, 3183, 2002, 2170, 1000, 1996, 7409, 1000, 1010, 1997, 9969, 4487, 23809, 3436, 2010, 3350, 1012, 102, 7727, 2000, 2032, 2004, 2069, 1000, 1996, 7409, 1000, 1010, 2572, 3217, 5831, 5496, 2010, 2567, 1997, 9969, 4487, 23809, 3436, 2010, 3350, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

### tokenizer 출력 이해하기

tokenizer 출력에는 보통 아래 값들이 들어 있습니다.

- `input_ids`: 문장을 토큰 ID 숫자로 바꾼 결과
- `attention_mask`: 실제 문장 토큰은 1, padding은 0으로 표시하는 값
- `token_type_ids`: 첫 번째 문장과 두 번째 문장을 구분하는 값

즉 모델은 원래 문장을 직접 보는 것이 아니라, `input_ids` 같은 숫자 배열을 입력으로 받습니다.

In [11]:
# input_ids는 숫자라서 바로 보면 문장처럼 읽기 어렵습니다.
# convert_ids_to_tokens를 사용하면 숫자 ID가 어떤 토큰인지 다시 확인할 수 있습니다.

tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])

print(tokens)

['[CLS]', 'am', '##ro', '##zi', 'accused', 'his', 'brother', ',', 'whom', 'he', 'called', '"', 'the', 'witness', '"', ',', 'of', 'deliberately', 'di', '##stor', '##ting', 'his', 'evidence', '.', '[SEP]', 'referring', 'to', 'him', 'as', 'only', '"', 'the', 'witness', '"', ',', 'am', '##ro', '##zi', 'accused', 'his', 'brother', 'of', 'deliberately', 'di', '##stor', '##ting', 'his', 'evidence', '.', '[SEP]']


In [12]:
# 숫자로 바뀐 결과의 길이를 확인합니다.
# 모델 입장에서는 이 숫자 배열이 실제 입력입니다.

print('input_ids 길이:', len(encoded['input_ids']))
print('attention_mask 길이:', len(encoded['attention_mask']))

if 'token_type_ids' in encoded:
    print('token_type_ids 길이:', len(encoded['token_type_ids']))

input_ids 길이: 50
attention_mask 길이: 50
token_type_ids 길이: 50


### 여기까지 정리

이번 단계에서 확인한 내용은 다음과 같습니다.

1. tokenizer는 문장을 토큰 단위로 나눕니다.
2. 각 토큰을 모델 vocabulary에 있는 숫자 ID로 바꿉니다.
3. 모델은 원문이 아니라 `input_ids`, `attention_mask` 같은 숫자 입력을 받습니다.

다음 단계에서는 데이터셋 전체에 tokenizer를 적용하는 방법을 배웁니다.

## STEP 4. 데이터셋 전체에 tokenizer 적용하기

STEP 3에서는 데이터 1개만 tokenizer에 넣어봤습니다.

하지만 모델을 학습하려면 train, validation, test 데이터 전체가 모델 입력 형태로 바뀌어 있어야 합니다.

Hugging Face `Dataset`은 `.map()`을 사용해서 데이터셋 전체에 같은 함수를 적용할 수 있습니다.

In [13]:
# 데이터셋의 각 batch에 tokenizer를 적용하는 함수를 만듭니다.
# batch에는 sentence1, sentence2 같은 컬럼이 여러 개 묶여 들어옵니다.

def tokenize_mrpc(batch):
    return tokenizer(
        batch['sentence1'],
        batch['sentence2'],
        truncation=True
    )

In [14]:
# map은 데이터셋 전체에 함수를 적용합니다.
# batched=True를 사용하면 샘플을 하나씩 처리하지 않고 여러 개씩 묶어서 처리하므로 더 빠릅니다.

tokenized_mrpc = mrpc.map(tokenize_mrpc, batched=True)

tokenized_mrpc

Map: 100%|██████████| 1725/1725 [00:00<00:00, 5442.71 examples/s]


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [ ]:
# tokenizer를 적용한 뒤 컬럼이 어떻게 바뀌었는지 확인합니다.
# 원래 있던 sentence1, sentence2, label, idx에 tokenizer 출력 컬럼이 추가됩니다.

print(tokenized_mrpc['train'].column_names)
tokenized_mrpc['train'][0]

### 여기까지 정리

이번 단계에서 한 일은 간단합니다.

1. 문장 두 개를 tokenizer에 넣는 함수를 만들었습니다.
2. `.map()`으로 그 함수를 데이터셋 전체에 적용했습니다.
3. 그 결과 `input_ids`, `attention_mask` 같은 모델 입력 컬럼이 데이터셋에 추가되었습니다.

다음 단계에서는 tokenizer가 적용된 데이터를 모델 학습에 사용할 수 있도록 정리합니다.

## STEP 5. 학습에 사용할 데이터 형태로 정리하기

tokenizer를 적용한 데이터셋에는 원래 문장 컬럼과 tokenizer 결과 컬럼이 함께 들어 있습니다.

모델 학습에는 원문 문장 자체보다 `input_ids`, `attention_mask`, `label` 같은 값이 필요합니다.

이번 단계에서는 Trainer가 바로 사용할 수 있도록 데이터셋의 형태를 정리합니다.

In [15]:
# 현재 tokenized_mrpc의 컬럼을 다시 확인합니다.
# sentence1, sentence2는 사람이 보기 위한 원문이고,
# input_ids, attention_mask는 모델이 실제로 사용할 숫자 입력입니다.

tokenized_mrpc['train'].column_names

['sentence1',
 'sentence2',
 'label',
 'idx',
 'input_ids',
 'token_type_ids',
 'attention_mask']

In [16]:
# 학습에 직접 쓰지 않을 원문 컬럼을 제거합니다.
# label은 정답이므로 남겨둡니다.

columns_to_remove = ['sentence1', 'sentence2', 'idx']

train_dataset = tokenized_mrpc['train'].remove_columns(columns_to_remove)
valid_dataset = tokenized_mrpc['validation'].remove_columns(columns_to_remove)
test_dataset = tokenized_mrpc['test'].remove_columns(columns_to_remove)

train_dataset

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3668
})

In [17]:
# 정리된 데이터 하나를 확인합니다.
# 이제 문장 텍스트 대신 숫자 입력과 label만 남아 있어야 합니다.

train_dataset[0]

{'label': 1,
 'input_ids': [101,
  2572,
  3217,
  5831,
  5496,
  2010,
  2567,
  1010,
  3183,
  2002,
  2170,
  1000,
  1996,
  7409,
  1000,
  1010,
  1997,
  9969,
  4487,
  23809,
  3436,
  2010,
  3350,
  1012,
  102,
  7727,
  2000,
  2032,
  2004,
  2069,
  1000,
  1996,
  7409,
  1000,
  1010,
  2572,
  3217,
  5831,
  5496,
  2010,
  2567,
  1997,
  9969,
  4487,
  23809,
  3436,
  2010,
  3350,
  1012,
  102],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

### 여기까지 정리

이번 단계에서는 학습용 데이터셋을 정리했습니다.

- 제거한 컬럼: `sentence1`, `sentence2`, `idx`
- 남긴 컬럼: `input_ids`, `attention_mask`, `label`, 필요하면 `token_type_ids`

이제 데이터는 모델에 들어갈 준비가 거의 끝났습니다.

다음 단계에서는 분류 모델을 불러옵니다.

## STEP 6. 분류 모델 불러오기

이제 문장 데이터를 숫자로 바꾸었으니, 그 숫자 입력을 받아 정답을 예측할 모델을 불러옵니다.

MRPC는 두 문장이 같은 의미인지 아닌지 맞히는 이진 분류 문제입니다.

따라서 출력 라벨 개수는 2개입니다.

- `0`: not_equivalent
- `1`: equivalent

In [18]:
from transformers import AutoModelForSequenceClassification

# AutoModelForSequenceClassification은 문장 분류용 모델을 불러옵니다.
# num_labels=2는 정답 종류가 2개라는 뜻입니다.
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

model

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6439.30it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


### 모델 출력층 이해하기

`distilbert-base-uncased`는 원래 범용 영어 문장 이해 모델입니다.

우리는 여기에 문장 분류용 출력층을 붙여서 MRPC 문제에 맞게 사용합니다.

처음 모델을 불러올 때 분류층 일부가 새로 초기화되었다는 경고가 나올 수 있습니다. 이것은 정상입니다.

아직 MRPC에 맞게 학습되지 않은 분류층을 앞으로 fine-tuning으로 학습시키면 됩니다.

In [19]:
# 모델 설정에서 라벨 개수를 확인합니다.

print('num_labels:', model.config.num_labels)
print('id2label:', model.config.id2label)

num_labels: 2
id2label: {0: 'LABEL_0', 1: 'LABEL_1'}


### 여기까지 정리

이번 단계에서는 분류 모델을 불러왔습니다.

- tokenizer: 문장을 숫자로 바꾸는 도구
- model: 숫자 입력을 보고 label을 예측하는 신경망
- `num_labels=2`: 정답이 2종류인 분류 문제라는 뜻

다음 단계에서는 Trainer를 사용해서 짧게 학습을 실행해봅니다.

## STEP 7. Trainer로 짧게 학습하고 평가하기

이제 데이터와 모델이 준비되었으므로 Hugging Face `Trainer`로 학습을 실행해봅니다.

전체 데이터를 오래 학습하기보다, 실습에서는 작은 subset으로 학습 흐름만 먼저 확인합니다.

실제 프로젝트에서는 같은 구조를 사용하되 데이터와 epoch 수를 늘려 성능을 개선합니다.

In [20]:
import numpy as np
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

# DataCollatorWithPadding은 batch 안에서 가장 긴 문장 길이에 맞춰 padding을 자동으로 넣어줍니다.
# 이렇게 하면 모든 샘플을 미리 같은 길이로 맞추지 않아도 됩니다.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [21]:
# 빠른 실습을 위해 일부 데이터만 사용합니다.
# 전체 학습 흐름을 확인하는 것이 목적입니다.

small_train_dataset = train_dataset.select(range(256))
small_valid_dataset = valid_dataset.select(range(128))

print('small train:', len(small_train_dataset))
print('small valid:', len(small_valid_dataset))

small train: 256
small valid: 128


In [22]:
# 평가 지표를 계산하는 함수를 만듭니다.
# predictions는 각 label에 대한 점수이므로, 가장 큰 점수를 가진 label을 예측값으로 사용합니다.

def compute_accuracy(eval_pred):
    predictions, labels = eval_pred
    predicted_labels = np.argmax(predictions, axis=1)
    accuracy = (predicted_labels == labels).mean()
    return {'accuracy': accuracy}

In [23]:
# Trainer가 사용할 학습 설정입니다.
# 여기서는 빠른 실습을 위해 epoch 수와 batch size를 작게 잡습니다.

training_args = TrainingArguments(
    output_dir='./hf_practice_mrpc',
    eval_strategy='epoch',
    save_strategy='no',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to='none'
)

In [25]:
# Trainer는 학습 루프를 대신 관리해줍니다.
# 데이터로더 생성, loss 계산, optimizer 업데이트, 평가 등을 묶어서 처리합니다.

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_accuracy
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.594128,0.703125


TrainOutput(global_step=32, training_loss=0.6095982789993286, metrics={'train_runtime': 2.2548, 'train_samples_per_second': 113.536, 'train_steps_per_second': 14.192, 'total_flos': 4681894621248.0, 'train_loss': 0.6095982789993286, 'epoch': 1.0})

In [26]:
# 학습이 끝난 뒤 validation 데이터로 성능을 확인합니다.

trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy
No log,0.594128,1,0.703125


{'eval_loss': 0.5941283702850342, 'eval_accuracy': 0.703125}

### 전체 실습 정리

이번 실습에서 Hugging Face의 기본 흐름을 한 번 끝까지 확인했습니다.

1. `datasets`로 데이터셋을 불러왔습니다.
2. `tokenizer`로 문장을 숫자로 바꿨습니다.
3. `AutoModelForSequenceClassification`으로 분류 모델을 불러왔습니다.
4. `Trainer`로 짧게 학습하고 평가했습니다.

이제 프로젝트에서는 같은 흐름을 MRPC가 아니라 NSMC 데이터셋과 `klue/bert-base` 모델로 바꾸면 됩니다.